In [ ]:
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [ ]:
model_path = "face_landmarker.task"


In [ ]:
import cv2
import mediapipe as mp 

In [ ]:
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions

In [ ]:
# try blendshapes 
# https://ai.google.dev/edge/mediapipe/solutions/vision/face_landmarker/python
options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    num_faces=1,
    output_face_blendshapes=True
)

landmarker = FaceLandmarker.create_from_options(options)

In [ ]:
cap = cv2.VideoCapture(0)

In [ ]:
from pythonosc.udp_client import SimpleUDPClient

while True:
    ret, frame = cap.read()
    if not ret:
        break

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=frame
    )

    result = landmarker.detect(mp_image)

    if result.face_blendshapes:
        blendshapes = result.face_blendshapes[0]

        # Convert to dictionary (IMPORTANT)
        data = {b.category_name: b.score for b in blendshapes}

        # Print a few useful ones
        print(
            "brow:", data.get("browInnerUp", 0),
            "smile:", data.get("mouthSmileLeft", 0),
            "jaw:", data.get("jawOpen", 0)
        )
        
        client.send_message("/face/brow", data.get("browInnerUp", 0))
        client.send_message("/face/jaw", data.get("jawOpen", 0))
        client.send_message("/face/smile", data.get("mouthSmileLeft", 0))

    cv2.imshow("cam", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
#pip install python-osc

In [ ]:
import cv2
import mediapipe as mp
from pythonosc.udp_client import SimpleUDPClient

#SETUP

client = SimpleUDPClient("127.0.0.1", 8000)

# MediaPipe setup
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='face_landmarker.task'),
    output_face_blendshapes=True,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

cap = cv2.VideoCapture(1)

# MAIN LOOP
import pynput
from pynput import keyboard
listener = keyboard.Listener(on_press=on_press)
listener.start()
def on_press(key):
    if key == keyboard.Key.esc:
        return True
    
if on_press:
    ret, frame = cap.read()
    # if not ret:
    #     break

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=frame
    )

    result = landmarker.detect(mp_image)

    if result.face_blendshapes:
        blendshapes = result.face_blendshapes[0]

        data = {b.category_name: b.score for b in blendshapes}

        # brow = data.get("browInnerUp", 0)
        # jaw = data.get("jawOpen", 0)
        # smile = data.get("mouthSmileLeft", 0)

        # print("brow:", brow, "jaw:", jaw, "smile:", smile)
        # print(data)

        #----------------------------
        # SET STATES
        #-----------------------------
        valence = data["mouthSmileLeft"] + data["mouthSmileRight"] - data["mouthFrownLeft"] - data["mouthFrownRight"]
        thinking = data["eyeLookUpLeft"] + data["eyeLookUpRight"]
        arousel = data["jawOpen"] + data["eyeWideLeft"] + data["eyeWideRight"] + data["browInnerUp"]
        anxious = data["eyeSquintLeft"] + data["eyeSquintRight"] + data["browDownLeft"] + data["browDownRight"] + data["mouthPressLeft"] + data["mouthPressRight"]
    
        #----------------------------
        # TEST CLICK AND GET RESULT 
        #-----------------------------



        #-----------------------------
        # send OSC messages
        #-----------------------------
        # client.send_message("/face/brow", brow)
        # client.send_message("/face/jaw", jaw)
        # client.send_message("/face/smile", smile)
        # landmarks = result.face_landmarks[0]
        # client.send_message("/pos/brow_x", landmarks[65].x)
        # client.send_message("/pos/brow_y", landmarks[65].y)

    # cv2.imshow("cam", frame)

    if cv2.waitKey(1) & 0xFF == 27:

        break

cap.release()
cv2.destroyAllWindows()

In [2]:
import cv2
import mediapipe as mp
from pythonosc.udp_client import SimpleUDPClient

#SETUP

client = SimpleUDPClient("127.0.0.1", 8000)

# MediaPipe setup
BaseOptions = mp.tasks.BaseOptions
FaceLandmarker = mp.tasks.vision.FaceLandmarker
FaceLandmarkerOptions = mp.tasks.vision.FaceLandmarkerOptions

options = FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path='face_landmarker.task'),
    output_face_blendshapes=True,
    num_faces=1
)

landmarker = FaceLandmarker.create_from_options(options)

cap = cv2.VideoCapture(1)

In [ ]:
mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=frame
    )

    result = landmarker.detect(mp_image)

    if result.face_blendshapes:
        blendshapes = result.face_blendshapes[0]

        data = {b.category_name: b.score for b in blendshapes}


In [1]:
import pynput
from pynput import keyboard

capture_now = False

def on_press(key):
    global capture_now

    if key == keyboard.Key.space:   # press SPACE to capture
        capture_now = True
        print("📸 Capture triggered")

    if key == keyboard.Key.esc:
        return False  # stop listener

listener = keyboard.Listener(on_press=on_press)
listener.start()

In [ ]:
while True:
    ret, frame = cap.read()
    if not ret:
        break

    if capture_now:
        capture_now = False  # reset flag

        print("Processing frame...")

        mp_image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=frame
        )

        result = landmarker.detect(mp_image)

        if result.face_blendshapes:
            blendshapes = result.face_blendshapes[0]
            data = {b.category_name: b.score for b in blendshapes}

            # ----------------------------
            # STATES
            # ----------------------------
            valence = data["mouthSmileLeft"] + data["mouthSmileRight"] - data["mouthFrownLeft"] - data["mouthFrownRight"]
            thinking = data["eyeLookUpLeft"] + data["eyeLookUpRight"]
            arousal = data["jawOpen"] + data["eyeWideLeft"] + data["eyeWideRight"] + data["browInnerUp"]
            anxious = data["eyeSquintLeft"] + data["eyeSquintRight"] + data["browDownLeft"] + data["browDownRight"] + data["mouthPressLeft"] + data["mouthPressRight"]
    
            # get suboptimal results 

            print("------ RESULT ------")
            print("Valence:", round(valence, 3))
            print("Thinking:", round(thinking, 3))
            print("Arousal:", round(arousal, 3))
            print("Anxious:", round(anxious, 3))

    cv2.imshow("cam", frame)

    if cv2.waitKey(1) & 0xFF == 27:
        break

📸 Capture triggered
Processing frame...
------ RESULT ------
Valence: -0.008
Thinking: 0.049
Arousal: 0.841
Anxious: 0.429
📸 Capture triggered
Processing frame...
------ RESULT ------
Valence: -0.03
Thinking: 0.039
Arousal: 0.508
Anxious: 0.597
📸 Capture triggered
Processing frame...
------ RESULT ------
Valence: -0.267
Thinking: 0.542
Arousal: 0.867
Anxious: 0.461
📸 Capture triggered
Processing frame...
------ RESULT ------
Valence: -0.027
Thinking: 0.033
Arousal: 0.504
Anxious: 0.57
📸 Capture triggered
Processing frame...
------ RESULT ------
Valence: 0.0
Thinking: 0.195
Arousal: 0.584
Anxious: 0.886


KeyboardInterrupt: 